In [0]:
%run ../config/config

In [0]:
!pip install jinja2==3.1.6

In [0]:
import json
import pandas as pd
import numpy as np
import os
import sys
import time

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import NumericType
from datetime import datetime

sys.path.insert(0, str(project_path))

from utils.data_quality import DataQualityAnalyzer, quality
from utils.reporting import HTMLReportGenerator, ChartGenerator, TableBuilder, HTMLFormatter
from utils.logger import MLOpsLogger, log_dataframe_info
from utils.evolutive_batch import (
    compute_evolutive_batch,
    build_evolutive_pivots,
    compute_historical_averages,
    compute_evolutive_semaphores,
    compute_evolutive_summary_kpis
)

print("Imports completados exitosamente")

In [0]:
logger = MLOpsLogger(
    name='06_quality_outputs',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={'mes_vta': mes_vta, 'environment': env, 'notebook': 'engine_quality'}
)

In [0]:
# =============================================================================
# CONFIGURACIÓN CENTRALIZADA
# =============================================================================
try:
    logger.info("=" * 60)
    logger.info("CARGANDO CONFIGURACIÓN CENTRALIZADA")
    logger.info("=" * 60)

    from utils.config_loader import ConfigLoader
    config_loader = ConfigLoader(quality_config)

    all_configs = config_loader.get_all_output_quality_config()

    analysis_settings = all_configs['analysis']
    column_settings = all_configs['columns']
    chart_settings = all_configs['charts']
    report_settings = all_configs['reports']
    semaforo_thresholds = all_configs['quality_thresholds']
    evolutive_semaforo_thresholds = all_configs['evolutive_thresholds']

    meses_historicos_calidad = analysis_settings.get('meses_historicos_calidad', MESES_HISTORICOS_CALIDAD)
    rows_per_page_config = analysis_settings.get('rows_per_page', 25)
    campo_primario = column_settings.get('campo_primario', 'codclaveunicocli')
    columnas_control = column_settings.get('columnas_control', [
        'codclaveunicocli', 'codclavepartycli', 'codmes',
        'codmes_2', 'codmes_1', 'fecrutina', 'fecactualizacionregistro', 'fecnacimiento'
    ])

    # Columnas raíz fuera del struct que también se analizan (ej: numpd)
    columnas_fijas_raiz = column_settings.get('columnas_fijas_raiz', ['numpd'])

    chart_figsize = tuple(chart_settings.get('figsize', [10, 6]))
    chart_bins = chart_settings.get('bins', 30)
    chart_dpi = chart_settings.get('dpi', 100)
    titulo_quality_report = report_settings.get('titulo_quality', 'Reporte de Calidad de Datos Outputs')

    logger.info(f"✓ CONFIGURACIÓN CENTRALIZADA CARGADA")
    logger.info(f"  Columnas fijas raíz: {columnas_fijas_raiz}")

except Exception as e:
    logger.log_exception(operation='load_centralized_config', exception=e, should_raise=True)

In [0]:
# =============================================================================
# ★ CARGA DESDE DELTA STAGING (plan depth = 1, sin RecursionError)
# =============================================================================
try:
    logger.log_stage_start('engine_quality', mes_vta=mes_vta, environment=env)
    start_time = time.time()

    spark = SparkSession.builder.getOrCreate()

    # ★ Path del staging creado por 01_extract_and_persist
    persist_path = f"{adls_location_table}/quality_engine/{env}/{TABLE_OUTPUT}"

    logger.info(f"Leyendo desde Delta staging: {persist_path}")
    load_start = time.time()

    # ★ Lectura directa desde Delta particionado - plan limpio
    df_staging = spark.read.format("delta").load(persist_path)

    # Filtrar mes actual
    df_actual = df_staging.filter(F.col("codmes") == mes_vta)
    actual_count = df_actual.count()

    if actual_count == 0:
        raise ValueError(f"No hay registros para el mes {mes_vta} en staging")

    # Filtrar histórico (todo lo que NO es mes actual)
    df_historico = df_staging.filter(F.col("codmes") < mes_vta)
    historico_count = df_historico.count()

    meses_historicos = sorted([r.codmes for r in df_historico.select('codmes').distinct().collect()])
    total_records = actual_count

    load_duration = time.time() - load_start
    logger.info(f"✓ Actual: {actual_count:,} | Histórico: {historico_count:,} ({len(meses_historicos)} meses) | {load_duration:.1f}s")
    logger.info(f"  ★ Plan depth = 1 (lectura directa desde Delta staging)")

except Exception as e:
    logger.log_exception(operation='load_from_staging', exception=e, should_raise=True)

In [0]:
# =============================================================================
# CONFIGURACIÓN DE VARIABLES
# =============================================================================
try:
    todas_columnas = df_actual.columns

    # Normalizar a minúsculas para comparaciones case-insensitive
    control_lower = {c.lower() for c in columnas_control}
    raiz_lower = {c.lower() for c in columnas_fijas_raiz}
    pk_lower = campo_primario.lower()

    # Variables del struct: todas las columnas analíticas excepto control, PK y fijas raíz
    variables_struct = [c for c in todas_columnas
                        if c.lower() not in control_lower
                        and c.lower() != pk_lower
                        and c.lower() not in raiz_lower]

    # Columnas raíz fijas (ej: numpd) – case-insensitive lookup
    col_lower_map = {c.lower(): c for c in todas_columnas}
    variables_raiz = [col_lower_map[r.lower()] for r in columnas_fijas_raiz
                      if r.lower() in col_lower_map]

    # Lista completa para el análisis de calidad
    variables_existentes = variables_raiz + variables_struct

    logger.info(f"Variables totales a analizar: {len(variables_existentes)}")
    logger.info(f"  Columnas raíz (fuera del struct): {len(variables_raiz)} → {variables_raiz}")
    logger.info(f"  Variables del struct aplanado: {len(variables_struct)}")

    if len(variables_existentes) == 0:
        raise ValueError("No hay variables para analizar")

except Exception as e:
    logger.log_exception(operation='configure_variables', exception=e, should_raise=True)

In [0]:
# =============================================================================
# EJECUCION DE CALIDAD (GLOBAL Y POR SEGMENTOS)
# =============================================================================
try:
    # Definir dictionary de segmentos a ejecutar
    segments_to_run = {"global": (df_actual, df_historico)}

    if USE_SEGMENTATION:
        for seg_name in MODELS_CONFIG.keys():
            df_act_seg = df_actual.filter(F.col(SEGMENTATION_COLUMN) == seg_name)
            df_hist_seg = df_historico.filter(F.col(SEGMENTATION_COLUMN) == seg_name)
            segments_to_run[seg_name] = (df_act_seg, df_hist_seg)

    for segment_name, (df_act, df_hist) in segments_to_run.items():
        logger.info("\n" + "=" * 80)
        logger.info(f"PROCESANDO CALIDAD PARA SEGMENTO: {segment_name.upper()}")
        logger.info("=" * 80)

        act_count = df_act.count()
        hist_count = df_hist.count()

        if act_count == 0:
            logger.warning(f"Saltando segmento {segment_name}: 0 registros actuales")
            continue

        # 1. Validaciones
        pk_stats = df_act.agg(
            F.sum(F.when(F.col(campo_primario).isNull(), 1).otherwise(0)).alias("null_keys"),
            F.count(campo_primario).alias("total"),
            F.approx_count_distinct(campo_primario).alias("approx_unique")
        ).collect()[0]

        if pk_stats["null_keys"] > 0:
            logger.warning(f"Llave primaria '{campo_primario}' tiene nulos en {segment_name}")

        # 2. Análisis Actual
        dq_actual = quality(df_act)
        if 'METRICS_CONFIG' not in globals():
            METRICS_CONFIG = None
        current_metrics = dq_actual.run(
            primary_key=campo_primario,
            columns=variables_existentes,
            semaforo_thresholds=semaforo_thresholds,
            metrics_config=METRICS_CONFIG
        )

        # 3. Evolutivo
        evolutive_metrics = {}
        evolutive_semaphore_results = None
        evolutive_kpis = None
        if hist_count > 0:
            full_metrics_df = compute_evolutive_batch(df_hist, variables_existentes)
            evolutive_metrics = build_evolutive_pivots(full_metrics_df)
            historical_averages = compute_historical_averages(full_metrics_df)
            evolutive_semaphore_results = compute_evolutive_semaphores(
                current_variable_metrics=current_metrics['variable_metrics'],
                historical_averages=historical_averages,
                evolutive_thresholds=evolutive_semaforo_thresholds,
                mes_vta=str(mes_vta)
            )
            evolutive_kpis = compute_evolutive_summary_kpis(evolutive_semaphore_results)

        # 4. Distribuciones
        if 'utils.metadata_serializer' in sys.modules:
            del sys.modules['utils.metadata_serializer']
        from utils.metadata_serializer import MetadataSerializer

        metadata_dir = os.path.join(temp_reports_input_path if 'inputs' in logger.name else temp_reports_output_path, "metadata", segment_name)
        os.makedirs(metadata_dir, exist_ok=True)
        serializer = MetadataSerializer(metadata_dir)

        df_act.createOrReplaceTempView(f"__tmp_dist_{segment_name}")
        distribution_data_path = serializer.serialize_distribution_data(
            spark=spark, table_name=f"__tmp_dist_{segment_name}",
            filter_sql=None, variables=variables_existentes,
            bins=chart_bins, filename="distribution_data.json"
        )

        # Score distribution by month (numpd + SC_* por codmes)
        try:
            from pyspark.sql.types import NumericType as _NType
            _schema_map = {f.name.lower(): f.dataType for f in df_act.schema.fields}
            _score_vars = [v for v in variables_existentes
                           if 'numpd' in v.lower() or v.upper().startswith('SC_')
                           or 'SCORE' in v.upper()]
            _score_vars = [v for v in _score_vars if isinstance(_schema_map.get(v.lower()), _NType)]
            _PD_CUTS = [0.01, 0.03, 0.07, 0.15, 0.30]

            _score_dist_by_month = {}
            for _var in _score_vars:
                _cuts = _PD_CUTS if 'numpd' in _var.lower() else []
                _pts = [0.0] + _cuts + [float('inf')]
                _bkt_expr = F.when(F.col(_var).isNull(), 'C0: NULL')
                for _ci in range(len(_pts) - 1):
                    _lo, _hi = _pts[_ci], _pts[_ci + 1]
                    _lbl = f'C{_ci+1}: [{_lo:.2f}, {"∞" if _hi==float("inf") else f"{_hi:.2f}"}]'
                    _cond = (F.col(_var) >= _lo) if _hi == float('inf') else ((F.col(_var) >= _lo) & (F.col(_var) < _hi))
                    _bkt_expr = _bkt_expr.when(_cond, _lbl)
                _bkt_expr = _bkt_expr.otherwise('C?')

                _all_labels = ['C0: NULL'] + [
                    f'C{_ci+1}: [{_pts[_ci]:.2f}, {"∞" if _pts[_ci+1]==float("inf") else f"{_pts[_ci+1]:.2f}"}]'
                    for _ci in range(len(_pts) - 1)]

                if hist_count > 0:
                    _cols = ['codmes', _var]
                    _df_all = df_act.select(*_cols).union(df_hist.select(*_cols))
                else:
                    _df_all = df_act.select('codmes', _var)

                _rows = _df_all.withColumn('bkt', _bkt_expr).groupBy('codmes', 'bkt').count().collect()
                _tots = _df_all.groupBy('codmes').count().collect()
                _tot_map = {str(r['codmes']): int(r['count']) for r in _tots}

                _by_month_raw = {}
                for _r in _rows:
                    _m = str(_r['codmes'])
                    if _m not in _by_month_raw:
                        _by_month_raw[_m] = {}
                    _by_month_raw[_m][_r['bkt']] = int(_r['count'])

                _by_period = {}
                for _m in sorted(_by_month_raw.keys()):
                    _tot = _tot_map.get(_m, 0)
                    _bkts = []
                    for _lbl in _all_labels:
                        _cnt = _by_month_raw[_m].get(_lbl, 0)
                        _pct = _cnt / _tot * 100 if _tot > 0 else 0
                        _bkts.append({'label': _lbl, 'count': _cnt, 'pct': round(_pct, 4)})
                    _by_period[_m] = {'total': _tot, 'buckets': _bkts}

                _score_dist_by_month[_var] = {'by_period': _by_period, 'cuts': _cuts}

            if _score_dist_by_month:
                import json as _sj
                _sp = os.path.join(metadata_dir, 'score_distribution_by_month.json')
                with open(_sp, 'w', encoding='utf-8') as _sf:
                    _sj.dump(_score_dist_by_month, _sf, ensure_ascii=False)
                logger.info(f"✓ score_distribution_by_month: {list(_score_dist_by_month.keys())} → {_sp}")
        except Exception as _se:
            logger.warning(f"score_distribution_by_month no se pudo computar: {_se}")

        # 5. Metadatos
        serializer.serialize_quality_metrics(current_metrics, filename="quality_metrics.json")
        serializer.serialize_evolutive_metrics(evolutive_metrics, filename="evolutive_metrics.json")
        if evolutive_semaphore_results is not None and not evolutive_semaphore_results.empty:
            serializer.serialize_quality_metrics(
                {'evolutive_semaphores': evolutive_semaphore_results, 'evolutive_kpis': evolutive_kpis},
                filename="evolutive_semaphores.json"
            )

        quality_bundle_path = serializer.save_metadata_bundle(
            mes_vta=mes_vta, environment=env, table_name=table_name_mt if 'inputs' in logger.name else output_table,
            total_records=act_count, variables_analyzed=variables_existentes,
            current_metrics=current_metrics, evolutive_metrics=evolutive_metrics,
            drift_results=None, variable_comparison=None,
            semaphore_kpis=None, semaphore_config=None,
            distribution_data_path=distribution_data_path,
            additional_metadata={'rows_per_page': rows_per_page_config, 'evolutive_kpis': evolutive_kpis if evolutive_kpis else {}},
            filename="quality_metadata.json"
        )
        logger.info(f"✓ Metadatos guardados para {segment_name}: {quality_bundle_path}")

    logger.log_stage_end('engine_quality', duration=time.time() - start_time, records_processed=total_records)

except Exception as e:
    logger.log_exception(operation='segment_processing', exception=e, should_raise=True)